# Solution E2 · Regression metrics computed wrongly: four bugs

**Type**: Find the bug

## Objective

Fix an evaluation function that runs without errors and prints plausible numbers, but computes the metrics the
wrong way, and rewrite it as a reusable function.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

df = ds.house_prices()
X = df[["area", "rooms", "distance_center_km"]]
y = df["price"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)

# the colleague trained the model on the logarithm of the price: there is no mistake here
model = LinearRegression().fit(X_tr, np.log(y_tr))

## Commented solution

Four bugs in five lines:

1. `r2_score(pred_log, y_te_log)` has the arguments swapped. R² is not symmetric: the denominator uses the
   variance of the first argument, which must be the **true** value.
2. Everything is computed on a logarithmic scale, while the business problem is in euros: "RMSE 0.16" means nothing
   to whoever looks at the prices.
3. `mean_squared_error` returns the **MSE**, not the RMSE: the "improvement" ratio is thus computed on the squares.
4. The baseline uses the mean of the **test set** (`y_te_log.mean()`): the test set is not touched to build the reference.

In [ ]:
def evaluate(y_true, y_pred, y_train):
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    baseline = np.full(len(y_true), np.mean(y_train))               # the mean of the TRAINING set
    rmse_base = mean_squared_error(y_true, baseline, squared=False)
    return {
        "r2": r2_score(y_true, y_pred),                              # (true, predicted), in this order
        "rmse": rmse,
        "rmse_baseline": rmse_base,
        "improvement_pct": 100 * (1 - rmse / rmse_base),
    }

pred_log = model.predict(X_te)
pred_euro = np.exp(pred_log)                                         # back to the unit of the target
result = evaluate(y_te, pred_euro, y_tr)
r2, rmse = result["r2"], result["rmse"]
rmse_baseline, improvement_pct = result["rmse_baseline"], result["improvement_pct"]
print(f"R2 {r2:.4f} | RMSE {rmse:,.0f} euros | baseline RMSE {rmse_baseline:,.0f} euros | improvement {improvement_pct:.1f}%")

In [ ]:
log_metrics = evaluate(y_te, np.exp(model.predict(X_te)), y_tr)
direct = LinearRegression().fit(X_tr, y_tr)
direct_metrics = evaluate(y_te, direct.predict(X_te), y_tr)
best_by_rmse = "log" if log_metrics["rmse"] < direct_metrics["rmse"] else "direct"

print(pd.DataFrame({"log": log_metrics, "direct": direct_metrics}).round(3).to_string())
print("best by RMSE:", best_by_rmse)

## Check

In [ ]:
with verify():
    check(close(r2, 0.92547, rtol=1e-4), "correct r2 (in euros, arguments in the right order)",
          "wrong r2: is it computed on the prices in euros? r2_score wants (y_true, y_pred) in this order")
    check(close(rmse, 32853.88, rtol=1e-4), "correct rmse (in euros, with the root)",
          "wrong rmse: it must be the root of the MSE, computed on the prices in euros")
    check(close(rmse_baseline, 120444.62, rtol=1e-4), "correct baseline (mean of the training set, in euros)",
          "wrong rmse_baseline: the baseline predicts the mean of y_tr, not that of the test set")
    check(close(improvement_pct, 72.72, rtol=1e-3), "correct improvement_pct", "improvement_pct = 100 * (1 - rmse / rmse_baseline)")
    check(set(log_metrics) == {"r2", "rmse", "rmse_baseline", "improvement_pct"}, "evaluate returns the four keys",
          "the dictionary must have r2, rmse, rmse_baseline, improvement_pct")
    check(close(log_metrics["rmse"], rmse), "evaluate reproduces the correct computation for the log model",
          "evaluate(y_te, np.exp(pred_log), y_tr) should give the same rmse")
    check(best_by_rmse == ("log" if log_metrics["rmse"] < direct_metrics["rmse"] else "direct"),
          "choice consistent with the numbers", "compare log_metrics['rmse'] and direct_metrics['rmse']")
done()

## What you learned

- The colleague's numbers (R² 0.941, "RMSE" 0.0251, "improvement" of 93.8%) look excellent and are all wrong:
  the right ones, in euros, are R² 0.925, RMSE 32,854 euros and an improvement of 72.7% over the baseline.
- The scikit-learn metrics want `(y_true, y_pred)` in this order. With MAE and MSE the error is hidden by the
  symmetry, with R² it is visible only by comparing with the correct value.
- An `evaluate(y_true, y_pred, y_train)` function reused for all the models eliminates the class of bugs "I computed the
  metrics in a different way for the two models".

## Common pitfall

Computing the baseline with the mean of the test set. The improvement looks a bit smaller or larger than the true one and the
difference is small: it is a bug that survives because it does not make anything collapse. The test set must not influence in
any way any reference quantity.

## Interview question

<details><summary>The model on the logarithm of the price has an R² of 0.94 computed by a script. The direct model has 0.92. Which one do you choose?</summary>

First I check where each R² is computed: if the one of the log model is computed on the logarithm of the price while
the other on the price, they are not comparable, they are two different targets. I would recompute both on the same target (the
price in euros, after bringing the predictions back with exp) and with the same function, and choose based on the metric
that reflects the cost of the error for the business.

</details>